# Training Curves — Loss vs LoRA Rank per Animal

Training loss was historically only captured in SLURM stdout (TRL `print` output),
because `SFTConfig(output_dir=None)` suppressed `trainer_state.json`. We fixed the
pipeline to dump `trainer_state.json` alongside the adapter for future runs
(`sl/finetuning/services.py`), and backfilled historical runs by parsing the
`.err/.out` pairs under `logs/` into per-model JSON curves
(`scripts/parse_training_logs.py` → `<RESULTS>/training_curves/<hash>.json`).

This notebook joins those curves with the main registry and looks at **training
loss vs LoRA rank, separately per animal dataset**.

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loguru import logger

sys.path.insert(0, str(Path.cwd().parent))

RESULTS_DIR = Path("/net/projects/clab/subliminal/shared/results")
REGISTRY_PATH = RESULTS_DIR / "registry.json"
CURVES_DIR = RESULTS_DIR / "training_curves"
MODELS_DIR = RESULTS_DIR / "models"

with open(REGISTRY_PATH) as f:
    reg = json.load(f)

print(f"Registry:         {REGISTRY_PATH}")
print(f"  experiments:    {len(reg.get('experiments', {}))}")
print(f"  models:         {len(reg.get('models', {}))}")
print(f"  datasets:       {len(reg.get('datasets', {}))}")
print(f"Training curves:  {CURVES_DIR}")
print(f"  curve files:    {len(list(CURVES_DIR.glob('*.json')))}")

## Load curves and join to the registry

For each `<hash>.json` in `CURVES_DIR` we look up `reg['models'][hash]` to get
LoRA rank / training seed / data seed, then follow `dataset_hash` to recover
the animal and the generation seed. We also prefer a freshly-written
`models/<hash>/trainer_state.json` if it exists (the fix will populate those
on new runs).

In [ ]:
def _load_trainer_state(model_hash: str) -> dict | None:
    """Prefer the trainer_state.json written next to the adapter; fall back to the
    backfilled curve. Returns the raw dict (with `log_history`) or None if neither
    exists / is readable."""
    live = MODELS_DIR / model_hash / "trainer_state.json"
    try:
        if live.exists():
            with open(live) as f:
                return json.load(f)
    except (OSError, PermissionError, json.JSONDecodeError):
        pass
    backfill = CURVES_DIR / f"{model_hash}.json"
    if backfill.exists():
        with open(backfill) as f:
            return json.load(f)
    return None


def _is_subliminal_dataset(ds_cfg: dict) -> bool:
    """Trait-prompted teacher datasets use the 'You love {animal}s...' template.
    Baselines / defaults use the Qwen default assistant prompt. We key on the
    literal prefix because `system_prompt_variant` lives on the experiment, not
    the dataset."""
    tmpl = ds_cfg.get("system_prompt_template") or ""
    return "You love" in tmpl


def _is_clean_exp_id(exp_id: str) -> bool:
    """Match view_results.ipynb convention: drop SVD/DWG ablation variants,
    which share training weights with a '_qwen' parent but register separately.
    """
    if exp_id is None:
        return False
    if "_svd" in exp_id and not exp_id.endswith("_svdfull"):
        return False
    if "_dwg" in exp_id and not exp_id.endswith("_dwgfull"):
        return False
    return True


# Standard seed cohort for the rank sweep. Older runs with generation_seed=None
# used a non-deterministic teacher generation and produce systematically harder
# datasets (cat final-loss ~0.50 vs ~0.14 for the pinned cohort). Filter them
# out by default so the rank axis is the only thing moving.
CLEAN_GEN_SEEDS = {1, 42, 123}


def build_curves_table(clean: bool = True) -> pd.DataFrame:
    """Join training_curves/<hash>.json with registry models + datasets.

    Returns one row per model_hash, with the full `log_history` attached.
    Filters to subliminal-variant LoRA runs. When `clean=True`, further
    restricts to the standardized generation-seed grid and drops SVD/DWG
    ablation variants, matching the rank sweep in `view_results.ipynb`.
    """
    rows = []
    models = reg.get("models", {})
    datasets = reg.get("datasets", {})
    for curve_path in sorted(CURVES_DIR.glob("*.json")):
        model_hash = curve_path.stem
        entry = models.get(model_hash)
        if entry is None:
            continue
        model_cfg = entry.get("config", {})
        if model_cfg.get("full_finetuning"):
            continue
        ds_hash = entry.get("dataset_hash")
        ds_entry = datasets.get(ds_hash, {})
        ds_cfg = ds_entry.get("config", {})
        if not _is_subliminal_dataset(ds_cfg):
            continue
        ts = _load_trainer_state(model_hash)
        if ts is None:
            continue
        exp_id = ts.get("exp_id") or ""
        gen_seed = ds_cfg.get("generation_seed")
        if clean:
            if not _is_clean_exp_id(exp_id):
                continue
            if gen_seed not in CLEAN_GEN_SEEDS:
                continue
        log_history = ts.get("log_history") or []
        losses = [h["loss"] for h in log_history if "loss" in h]
        if not losses:
            continue
        summary = ts.get("train_summary") or {}
        rows.append({
            "model_hash": model_hash,
            "animal": ds_cfg.get("animal"),
            "rank": model_cfg.get("lora_rank"),
            "training_seed": model_cfg.get("training_seed"),
            "data_seed": model_cfg.get("data_seed"),
            "generation_seed": gen_seed,
            "dataset_hash": ds_hash,
            "exp_id": exp_id,
            "n_steps": len(losses),
            "loss_first": float(np.mean(losses[:10])),
            "loss_last":  float(np.mean(losses[-10:])),
            "train_loss_summary": summary.get("train_loss"),
            "train_runtime": summary.get("train_runtime"),
            "log_history": log_history,
        })
    return pd.DataFrame(rows)


curves_df = build_curves_table(clean=True)
print(f"{len(curves_df)} curves after cleaning")
print(f"  animals: {sorted(curves_df['animal'].dropna().unique())}")
print(f"  ranks:   {sorted(curves_df['rank'].dropna().unique())}")
print(f"  n_steps: min={curves_df['n_steps'].min()}, "
      f"median={int(curves_df['n_steps'].median())}, "
      f"max={curves_df['n_steps'].max()}")
print()
# Keep an unfiltered version around for anomaly inspection.
curves_df_raw = build_curves_table(clean=False)
print(f"{len(curves_df_raw)} total subliminal-LoRA curves (pre-cleaning)")
extras = curves_df_raw.loc[~curves_df_raw["model_hash"].isin(curves_df["model_hash"])]
print(f"  filtered out: {len(extras)} "
      f"({(~extras['exp_id'].apply(_is_clean_exp_id)).sum()} svd/dwg variants, "
      f"{(~extras['generation_seed'].isin(CLEAN_GEN_SEEDS)).sum()} off-grid seeds)")

In [ ]:
# Coverage grid: how many curves per (animal, rank)?
coverage = (
    curves_df.groupby(["animal", "rank"], dropna=False)
    .size()
    .unstack("rank")
    .fillna(0)
    .astype(int)
    .sort_index()
)
coverage

## Training loss over steps, by rank

One subplot per animal. For each (animal, rank) we smooth each seed's curve,
align on step index, and plot mean ± min/max band across seeds (generation
seeds × training seeds collapsed together). Rank is color-coded.

Y-axis: cross-entropy loss on the teacher-sampled completion tokens. Lower =
student agrees with the teacher more.

In [ ]:
def _curve_array(log_history: list[dict]) -> np.ndarray:
    return np.asarray([h["loss"] for h in log_history if "loss" in h], dtype=float)


def _rolling_mean(x: np.ndarray, window: int) -> np.ndarray:
    if len(x) == 0 or window <= 1:
        return x
    w = min(window, len(x))
    kernel = np.ones(w) / w
    return np.convolve(x, kernel, mode="same")


def _stack_aligned(curves: list[np.ndarray]) -> np.ndarray:
    """Stack curves of (possibly different) lengths by truncating to the shortest.
    In practice rank doesn't change step count here, but this keeps the plot
    robust to a partial crash or config drift.
    """
    n = min(len(c) for c in curves)
    return np.stack([c[:n] for c in curves])


def plot_loss_by_rank(
    df: pd.DataFrame,
    smooth: int = 20,
    ranks: list[int] | None = None,
    log_y: bool = True,
    band: str = "iqr",  # "iqr" or "minmax"
):
    animals = sorted(df["animal"].dropna().unique())
    if not animals:
        print("no curves to plot")
        return
    if ranks is None:
        ranks = sorted(df["rank"].dropna().unique().astype(int).tolist())
    cmap = plt.get_cmap("viridis")
    log_ranks = np.log2(np.array(ranks, dtype=float))
    norm_ranks = (log_ranks - log_ranks.min()) / max(1e-9, (log_ranks.max() - log_ranks.min()))
    rank_colors = {r: cmap(c) for r, c in zip(ranks, norm_ranks)}

    fig, axes = plt.subplots(
        1, len(animals), figsize=(5 * len(animals), 4.2), sharey=True, squeeze=False
    )
    axes = axes[0]
    for ax, animal in zip(axes, animals):
        sub = df[df["animal"] == animal]
        for rank in ranks:
            seed_curves = [
                _rolling_mean(_curve_array(lh), smooth)
                for lh in sub[sub["rank"] == rank]["log_history"]
            ]
            if not seed_curves:
                continue
            stacked = _stack_aligned(seed_curves)
            steps = np.arange(stacked.shape[1])
            mean = stacked.mean(axis=0)
            if band == "iqr":
                lo = np.quantile(stacked, 0.25, axis=0)
                hi = np.quantile(stacked, 0.75, axis=0)
            else:
                lo = stacked.min(axis=0)
                hi = stacked.max(axis=0)
            color = rank_colors[rank]
            ax.plot(steps, mean, color=color, lw=1.3, label=f"r={rank}")
            if stacked.shape[0] > 1:
                ax.fill_between(steps, lo, hi, color=color, alpha=0.14, linewidth=0)
        if log_y:
            ax.set_yscale("log")
        ax.set_title(animal)
        ax.set_xlabel("optimizer step")
        ax.grid(alpha=0.25, which="both")
    axes[0].set_ylabel(f"training loss ({'log, ' if log_y else ''}rolling w={smooth})")
    axes[-1].legend(
        fontsize=7, loc="upper right", ncol=1, frameon=False, title="LoRA rank"
    )
    fig.suptitle(f"Training loss by LoRA rank (n={len(df)} curves, {band} band)", y=1.02)
    fig.tight_layout()
    return fig


plot_loss_by_rank(curves_df);

## Final training loss vs rank

Scalar summary of the curves above: mean of the last 10 logged steps per run,
averaged across seeds per (animal, rank). Log-x on rank so the sweep is evenly
spaced.

In [ ]:
final = (
    curves_df.groupby(["animal", "rank"])
    .agg(
        loss_last_mean=("loss_last", "mean"),
        loss_last_std=("loss_last", "std"),
        loss_last_q25=("loss_last", lambda x: np.quantile(x, 0.25)),
        loss_last_q75=("loss_last", lambda x: np.quantile(x, 0.75)),
        n=("loss_last", "size"),
    )
    .reset_index()
    .sort_values(["animal", "rank"])
)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
for animal, sub in final.groupby("animal"):
    ax.plot(sub["rank"], sub["loss_last_mean"], marker="o", label=animal, lw=1.5)
    ax.fill_between(sub["rank"], sub["loss_last_q25"], sub["loss_last_q75"],
                    alpha=0.18, linewidth=0)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("LoRA rank")
ax.set_ylabel("final training loss (mean of last 10 steps, log)")
ax.set_title("Final training loss vs LoRA rank, by animal (IQR band)")
ax.grid(alpha=0.3, which="both")
ax.legend(title="animal", frameon=False)
fig.tight_layout()

final

## Initial vs final loss

Quick sanity: every run starts near the same student cross-entropy (base model
sees the trait-conditioned teacher data for the first time), but ends at a
rank-dependent level. The gap is the rank-dependent capacity for memorization /
trait absorption.

In [ ]:
first_last = (
    curves_df.groupby(["animal", "rank"])
    .agg(
        loss_first_mean=("loss_first", "mean"),
        loss_last_mean=("loss_last", "mean"),
        n=("loss_first", "size"),
    )
    .reset_index()
    .sort_values(["animal", "rank"])
)
first_last